# Reason-Graph — FELM-Science reproduction with Llama-3-8B-Instruct

Runs our BTProp reproduction on Kaggle's free GPUs with the paper's own open-source model, its
own prompts, and its confidence method (real **True/False token probabilities**).

**Before running — notebook settings (right-hand panel):**
1. *Accelerator*: **GPU T4 x2** (the model is 16 GB; it's split across both GPUs).
2. *Internet*: **On** (needs a phone-verified Kaggle account).
3. *Add-ons → Secrets*: a secret named **`GITHUB_TOKEN`** (read-only token for the private repo),
   attached to this notebook.

Then **Save Version → Save & Run All**. It runs on Kaggle's servers (~5–7 h); you can close the
laptop. Results appear in the notebook's **Output** tab. Steps in `docs/project-log/07-running-on-kaggle.md`.

## 1. Settings

In [ ]:
BRANCH = "main"  # branch to run -- must be pushed to GitHub
MODEL = "NousResearch/Meta-Llama-3-8B-Instruct"

## 2. Check the GPUs (should list two Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Download our code from GitHub
The token is read from Kaggle Secrets, used once, then removed from the saved git config so it never
ends up in the notebook's output files.

In [ ]:
import os
import subprocess

from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
repo = "github.com/Mahakisore7/Reason-Graph.git"
os.chdir("/kaggle/working")
subprocess.run(
    ["git", "clone", "--quiet", "--branch", BRANCH, f"https://{token}@{repo}"], check=True
)
subprocess.run(
    ["git", "-C", "Reason-Graph", "remote", "set-url", "origin", f"https://{repo}"], check=True
)
del token
os.chdir("/kaggle/working/Reason-Graph")
!git log --oneline -3

## 4. Install what Kaggle doesn't already have
Kaggle already ships PyTorch with GPU support, scikit-learn and pandas. We add the rest, and make sure
`transformers` is recent enough for our code.

In [ ]:
!pip install -q tenacity openai sentence-transformers accelerate "transformers>=4.56"
import os

os.environ["PYTHONPATH"] = "/kaggle/working/Reason-Graph"

## 5. Make every sentence stand alone (all 595 segments, ~40 min)
Llama rewrites each FELM segment using its question and full answer as context ("They are located on
the wings" → "Ground spoilers are located on the wings"). A rewrite is kept only if the NLI model says
it still says everything the original said; otherwise the original is kept. Labels never change.

In [ ]:
!python -u scripts/decontextualize_felm.py --backend hf --model {MODEL}

## 6. Build the calibration (fit half: 291 segments)
For each segment, one forward pass of Llama reads P(True) vs P(False). Fits both the 5-bucket table
and the smooth curve. A few minutes.

In [ ]:
!python -u scripts/fit_emission_model.py --backend hf --model {MODEL} --standalone

## 7. Test on the held-out half (304 segments) — the long step
For every test segment: build a belief tree with the paper's prompts (decomposition / premises /
correction), label links with the NLI model, run the tree math, and report AUROC for raw confidence,
calibration alone, and calibration + tree. Everything is cached in `llm_cache/`.

In [ ]:
!python -u scripts/evaluate_felm.py --backend hf --model {MODEL} --standalone

## 8. Summary

In [ ]:
import glob
import json

for path in sorted(glob.glob("results/*.json")):
    with open(path) as f:
        data = json.load(f)
    keys = [
        "llm",
        "num_segments",
        "num_changed",
        "num_rejected",
        "num_examples",
        "sample_size",
        "auroc",
    ]
    print(path)
    for k in keys:
        if k in data:
            print(f"   {k}: {data[k]}")

## 9. Package results for download
`reason_graph_results.zip` (results + the LLM answer cache) appears in the Output tab. Unzip it into the
repo on your laptop to analyse without re-running anything.

In [ ]:
!cd /kaggle/working && zip -qr reason_graph_results.zip Reason-Graph/results Reason-Graph/llm_cache && ls -lh reason_graph_results.zip